# Music ML — Composer-Conditioned Piano Generation
### Training notebook for Google Colab (GPU)

**Before running:** make sure you selected a GPU runtime.
`Runtime → Change runtime type → T4 GPU`

## Step 1 — Mount Google Drive and unzip project

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# Path to the zip you uploaded to Drive (adjust if you put it in a subfolder)
ZIP_PATH = '/content/drive/MyDrive/music_ml_src.zip'
WORK_DIR = '/content/music_ml'

os.makedirs(WORK_DIR, exist_ok=True)
!unzip -q "{ZIP_PATH}" -d "{WORK_DIR}"

# Show what we have
!ls "{WORK_DIR}"

replace /content/music_ml/src/training/config.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: data  notebooks  requirements.txt  scripts  src  tests


In [20]:
# Move into the project root so all relative paths work
os.chdir(WORK_DIR)
import sys
sys.path.insert(0, WORK_DIR)
print('Working directory:', os.getcwd())

Working directory: /content/music_ml


## Step 1b — Patch src/ from Drive *(run this if src/ has been updated since you last uploaded the zip)*

If you updated `src/` locally (e.g. added theory/era support to transformer.py), zip just that folder and upload it to Drive as `music_ml_src.zip`, then run this cell to overwrite the old `src/` in the Colab session. Skip this step if your zip is already up to date.

In [21]:
import os, shutil

SRC_ZIP_ON_DRIVE = '/content/drive/MyDrive/music_ml_src.zip'
WORK_DIR         = '/content/music_ml'

if not os.path.exists(SRC_ZIP_ON_DRIVE):
    print(f'Skipping — {SRC_ZIP_ON_DRIVE} not found on Drive.')
    print('To use this cell:')
    print('  1. On your local machine, zip the src/ folder:')
    print('       cd "Music ML Project" && zip -r music_ml_src.zip src/')
    print('  2. Upload music_ml_src.zip to the root of your Google Drive.')
    print('  3. Re-run this cell.')
else:
    dst_src = os.path.join(WORK_DIR, 'src')
    if os.path.isdir(dst_src):
        shutil.rmtree(dst_src)
    !unzip -q "{SRC_ZIP_ON_DRIVE}" -d "{WORK_DIR}"
    print(f'src/ patched from {SRC_ZIP_ON_DRIVE}')
    print('Updated files:')
    for root, dirs, files in os.walk(dst_src):
        for f in files:
            print(f'  {os.path.relpath(os.path.join(root, f), WORK_DIR)}')

src/ patched from /content/drive/MyDrive/music_ml_src.zip
Updated files:
  src/__init__.py
  src/__pycache__/__init__.cpython-314.pyc
  src/__pycache__/__init__.cpython-311.pyc
  src/__pycache__/__init__.cpython-313.pyc
  src/training/__init__.py
  src/training/trainer.py
  src/training/config.py
  src/training/__pycache__/trainer.cpython-313.pyc
  src/training/__pycache__/config.cpython-311.pyc
  src/training/__pycache__/__init__.cpython-311.pyc
  src/training/__pycache__/__init__.cpython-313.pyc
  src/training/__pycache__/config.cpython-313.pyc
  src/data/__init__.py
  src/data/composer_meta.py
  src/data/theory_extractor.py
  src/data/preprocess.py
  src/data/dataset.py
  src/data/midi_parser.py
  src/data/__pycache__/midi_parser.cpython-313.pyc
  src/data/__pycache__/__init__.cpython-314.pyc
  src/data/__pycache__/midi_parser.cpython-314.pyc
  src/data/__pycache__/dataset.cpython-314.pyc
  src/data/__pycache__/preprocess.cpython-314.pyc
  src/data/__pycache__/composer_meta.cpython-

## Step 2 — Install dependencies

In [22]:
!pip install -q mido tqdm
# torch is pre-installed on Colab; just verify GPU is visible
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Step 3 — Verify processed data

In [9]:
import json, numpy as np

processed_dir = 'data/processed'
with open(os.path.join(processed_dir, 'composer_map.json')) as f:
    composer_map = json.load(f)

total_files = sum(
    len([x for x in os.listdir(os.path.join(processed_dir, c)) if x.endswith('.npy')])
    for c in composer_map if os.path.isdir(os.path.join(processed_dir, c))
)

print(f'Composers: {len(composer_map)}')
print(f'Processed files: {total_files}')
print('\nComposer map:')
for name, idx in sorted(composer_map.items(), key=lambda x: x[1]):
    d = os.path.join(processed_dir, name)
    n = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f'  {idx:2d}: {name} ({n} files)')

Composers: 43
Processed files: 1276

Composer map:
   0: Alban Berg (3 files)
   1: Alexander Scriabin (35 files)
   2: Antonio Soler (1 files)
   3: Carl Maria von Weber (1 files)
   4: Charles Gounod (1 files)
   5: Claude Debussy (45 files)
   6: César Franck (5 files)
   7: Domenico Scarlatti (31 files)
   8: Edvard Grieg (3 files)
   9: Felix Mendelssohn (31 files)
  10: Franz Liszt (134 files)
  11: Franz Schubert (197 files)
  12: Fritz Kreisler (1 files)
  13: Frédéric Chopin (201 files)
  14: George Enescu (1 files)
  15: George Frideric Handel (5 files)
  16: Georges Bizet (3 files)
  17: Giuseppe Verdi (1 files)
  18: Henry Purcell (1 files)
  19: Isaac Albéniz (8 files)
  20: Jean-Philippe Rameau (1 files)
  21: Johann Christian Fischer (1 files)
  22: Johann Pachelbel (1 files)
  23: Johann Sebastian Bach (156 files)
  24: Johann Strauss (1 files)
  25: Johannes Brahms (26 files)
  26: Joseph Haydn (40 files)
  27: Leoš Janáček (4 files)
  28: Ludwig van Beethoven (146 fil

## Step 4 — Configure training

In [ ]:
from src.training.config import TrainConfig

cfg = TrainConfig(
    processed_data_dir = 'data/processed',
    checkpoint_dir     = '/content/drive/MyDrive/music_ml_checkpoints',  # save directly to Drive

    # Model
    d_model            = 512,
    n_heads            = 8,
    n_layers           = 6,
    d_ff               = 2048,
    dropout            = 0.1,
    num_composers      = len(composer_map),
    composer_embed_dim = 64,

    # Training
    seq_len            = 512,
    stride             = 256,
    batch_size         = 32,
    num_epochs         = 100,
    learning_rate      = 1e-4,
    warmup_steps       = 4000,
    save_every         = 1,
    log_every          = 200,
)

os.makedirs(cfg.checkpoint_dir, exist_ok=True)
print('Checkpoint dir:', cfg.checkpoint_dir)
print('Config ready.')

## Step 5 — Build dataset and model

In [ ]:
from src.training.trainer import Trainer

trainer = Trainer(cfg, composer_map)
print('Ready to train.')

## Step 6 — Train

> Checkpoints are saved to your Google Drive every 5 epochs and whenever validation loss improves.  
> If the Colab session disconnects, re-run cells 1–5, then run **Step 6b** to resume from the last checkpoint.

In [ ]:
trainer.train()

## Step 6b — Resume from checkpoint (run this instead of Step 6 after a disconnect)

In [ ]:
import torch, glob, os

CHECKPOINT_DIR = '/content/drive/MyDrive/music_ml_checkpoints'

# Prioritize 'checkpoint_best.pt', then fallback to the latest epoch-based checkpoint
best_path = os.path.join(CHECKPOINT_DIR, 'checkpoint_best.pt')
epoch_ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'checkpoint_epoch*.pt')))

if os.path.exists(best_path):
    resume_path = best_path
elif epoch_ckpts:
    resume_path = epoch_ckpts[-1]
else:
    resume_path = None

if resume_path:
    print(f'Resuming from: {resume_path}')

    from src.training.trainer import Trainer
    from src.training.config import TrainConfig

    # Securely allow the config class
    torch.serialization.add_safe_globals([TrainConfig])

    ckpt = torch.load(resume_path, map_location='cpu')
    cfg_saved = ckpt['config']
    composer_map = ckpt['composer_map']
    start_epoch = ckpt['epoch']

    # Update paths and optimize DataLoader workers for Colab
    cfg_saved.checkpoint_dir = CHECKPOINT_DIR
    cfg_saved.processed_data_dir = 'data/processed'
    cfg_saved.num_workers = 2  # Address the DataLoader warning

    trainer = Trainer(cfg_saved, composer_map)
    trainer.model.load_state_dict(ckpt['model_state'])
    trainer.optimizer.load_state_dict(ckpt['optim_state'])
    print(f'Loaded checkpoint from epoch {start_epoch}. Continuing...')

    trainer.train()
else:
    print('No checkpoint found to resume from.')

## Step 7 — Generate a sample

Set `USE_THEORY_CHECKPOINT = True` to use the Phase 1.5 fine-tuned model (era conditioning + per-composer style constraints). Set to `False` to use the base Phase 1 model.

Upload your `checkpoint_theory_best.pt` from Kaggle to `music_ml_checkpoints/` on Google Drive before running.

**Priming (optional):** Set `PRIMER_MIDI` to a path on Drive pointing to a short MIDI excerpt (0.5–1.5 s, ~50–150 tokens) from the same composer. The generation will continue from that passage, inheriting its key, rhythm, and harmonic context. Leave `PRIMER_MIDI = None` to generate from scratch.

In [ ]:
import torch, os, glob
import torch.nn.functional as F
from src.model.transformer import MusicTransformer
from src.data.midi_parser import events_to_midi, midi_to_events, SOS_TOKEN, EOS_TOKEN, PAD_TOKEN
from src.training.config import TrainConfig

# ── Settings — edit these ─────────────────────────────────────────────────────
CHECKPOINT_DIR        = '/content/drive/MyDrive/music_ml_checkpoints'
COMPOSER              = 'Frédéric Chopin'   # any name from the composer map above

USE_THEORY_CHECKPOINT = True    # True → theory fine-tuned model; False → base model

# ── Priming (optional) ────────────────────────────────────────────────────────
# Set PRIMER_MIDI to a path on Drive (or /content/) to condition the generation
# on a real MIDI excerpt. Use a short passage (0.5–1.5 s) from the same composer
# for best results. Set to None to start from scratch.
PRIMER_MIDI       = None   # e.g. '/content/drive/MyDrive/primers/chopin_ballade.mid'
PRIMER_MAX_TOKENS = 128    # tokens kept from the primer (caps context usage)

MAX_TOKENS  = 4096
MIN_TOKENS  = 1024   # EOS suppressed until at least this many tokens are generated
TEMPERATURE = 1.15
TOP_K       = 50
TOP_P       = 0.95
MAX_RETRIES = 10

OUTPUT                = '/content/drive/MyDrive/music_ml_output/' + COMPOSER + ' temp=' + str(TEMPERATURE) + ' generated.mid'

# ── Select checkpoint ─────────────────────────────────────────────────────────
if USE_THEORY_CHECKPOINT:
    ckpt_path = os.path.join(CHECKPOINT_DIR, 'checkpoint_theory_best.pt')
    if not os.path.exists(ckpt_path):
        epoch_ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'checkpoint_theory_epoch*.pt')))
        if epoch_ckpts:
            ckpt_path = epoch_ckpts[-1]
            print(f'checkpoint_theory_best.pt not found — using {os.path.basename(ckpt_path)}')
        else:
            raise FileNotFoundError(
                'No theory checkpoint found. Download checkpoint_theory_best.pt '
                '(or checkpoint_theory_epoch*.pt) from Kaggle Output and upload it '
                'to music_ml_checkpoints/ on Google Drive.'
            )
else:
    ckpt_path = os.path.join(CHECKPOINT_DIR, 'checkpoint_best.pt')

print(f'Checkpoint : {ckpt_path}')
print(f'Composer   : {COMPOSER}')

# ── Load checkpoint and build model from saved config ────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.serialization.add_safe_globals([TrainConfig])

ckpt         = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg          = ckpt['config']
composer_map = ckpt['composer_map']

use_theory = getattr(cfg, 'use_theory', False)
use_era    = getattr(cfg, 'use_era',    False)
print(f'use_theory={use_theory}  use_era={use_era}')

model = MusicTransformer(
    vocab_size         = getattr(cfg, 'vocab_size', 392),
    d_model            = cfg.d_model,
    n_heads            = cfg.n_heads,
    n_layers           = cfg.n_layers,
    d_ff               = cfg.d_ff,
    dropout            = 0.0,
    max_seq_len        = getattr(cfg, 'max_seq_len', 1024),
    num_composers      = len(composer_map),
    composer_embed_dim = cfg.composer_embed_dim,
    use_theory         = use_theory,
    use_era            = use_era,
)
model.load_state_dict(ckpt['model_state'], strict=True)
model.to(device)
model.eval()
print(f'Model loaded — {sum(p.numel() for p in model.parameters()):,} parameters')

if COMPOSER not in composer_map:
    raise ValueError(f'"{COMPOSER}" not in composer map.\nAvailable: {sorted(composer_map.keys())}')

# ── Load primer (optional) ────────────────────────────────────────────────────
primer_tokens = [SOS_TOKEN]
if PRIMER_MIDI is not None:
    if not os.path.exists(PRIMER_MIDI):
        raise FileNotFoundError(f'PRIMER_MIDI not found: {PRIMER_MIDI}')
    raw = midi_to_events(PRIMER_MIDI)
    # midi_to_events returns [SOS, ...events..., EOS] — keep only the inner events
    inner = [t for t in raw[1:] if t != EOS_TOKEN]
    inner = inner[:PRIMER_MAX_TOKENS]
    primer_tokens = [SOS_TOKEN] + inner
    note_count_primer = sum(1 for t in inner if 0 <= t <= 127)
    print(f'Primer     : {len(inner)} tokens, {note_count_primer} NOTE_ON events from {os.path.basename(PRIMER_MIDI)}')
else:
    print('Primer     : none (generating from scratch)')

# ── Self-contained generation loop (no dependency on generate.py version) ─────
def _top_k_top_p(logits, top_k, top_p):
    if top_k > 0:
        thresh = torch.topk(logits, min(top_k, logits.size(-1))).values[..., -1, None]
        logits = logits.masked_fill(logits < thresh, float('-inf'))
    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        remove = (cum_probs - F.softmax(sorted_logits, dim=-1)) > top_p
        sorted_logits[remove] = float('-inf')
        logits = torch.zeros_like(logits).scatter_(-1, sorted_idx, sorted_logits)
    return logits

def _generate_piece(model, composer_id, device, max_tokens, min_tokens,
                    temperature, top_k, top_p, context_window,
                    composer_name, use_era, use_theory=False,
                    primer_tokens=None):
    model.eval()
    from src.data.composer_meta import get_era_id
    from src.data.theory_extractor import KEY_UNKNOWN, CHORD_UNKNOWN, BEAT_UNKNOWN

    c_tensor   = torch.tensor([composer_id], dtype=torch.long, device=device)
    era_tensor = None
    if use_era and composer_name is not None:
        era_tensor = torch.tensor([get_era_id(composer_name)], dtype=torch.long, device=device)

    _constraint_fn = None
    try:
        from src.inference.composer_constraints import build_constraint_fn
        if composer_name is not None:
            _constraint_fn = build_constraint_fn(composer_name)
    except Exception:
        pass

    # Start from primer if provided, otherwise just SOS
    history  = list(primer_tokens) if primer_tokens else [SOS_TOKEN]
    max_ctx  = context_window
    # Tokens already in the primer don't count toward min_tokens
    primer_len = len(history) - 1  # exclude SOS itself

    with torch.no_grad():
        for _ in range(max_tokens):
            window = history[-max_ctx:]
            x      = torch.tensor([window], dtype=torch.long, device=device)
            T      = x.size(1)

            if use_theory:
                key_t   = torch.full((1, T), KEY_UNKNOWN,   dtype=torch.long, device=device)
                chord_t = torch.full((1, T), CHORD_UNKNOWN, dtype=torch.long, device=device)
                beat_t  = torch.full((1, T), BEAT_UNKNOWN,  dtype=torch.long, device=device)
            else:
                key_t = chord_t = beat_t = None

            logits = model(x, c_tensor, era_ids=era_tensor,
                           key_ids=key_t, chord_ids=chord_t, beat_ids=beat_t)
            next_logits = logits[0, -1, :] / max(temperature, 1e-8)

            next_logits[PAD_TOKEN] = float('-inf')
            next_logits[SOS_TOKEN] = float('-inf')
            # Count only tokens generated after the primer toward min_tokens
            generated_so_far = len(history) - 1 - primer_len
            if generated_so_far < min_tokens:
                next_logits[EOS_TOKEN] = float('-inf')
            else:
                tokens_remaining = max_tokens - generated_so_far
                if tokens_remaining < 256:
                    try:
                        from src.inference.composer_constraints import (
                            _estimate_key, _tonic_triad_pcs,
                        )
                        key_id = _estimate_key(history)
                        if key_id is not None:
                            fade  = 1.0 - tokens_remaining / 256
                            boost = 3.5 * fade
                            tonic = _tonic_triad_pcs(key_id)
                            for p in range(128):
                                if p % 12 in tonic:
                                    next_logits[p] += boost
                            next_logits[EOS_TOKEN] += boost * 0.8
                    except Exception:
                        pass

            if not any(0 <= t <= 127 for t in history):
                drought = len(history) - 1
                if drought >= 8:
                    next_logits[:128] += min(0.15 * (drought - 8), 2.5)

            if _constraint_fn is not None:
                next_logits = _constraint_fn(next_logits, history)

            next_logits = _top_k_top_p(next_logits, top_k, top_p)
            probs       = F.softmax(next_logits, dim=-1)
            next_token  = torch.multinomial(probs, num_samples=1).item()

            if next_token == EOS_TOKEN:
                break
            history.append(next_token)

    return history

# ── Post-process: close held notes and add final silence ─────────────────────
def _finalize_tokens(tokens):
    """Close held notes so nothing hangs open after the piece ends."""
    active = set()
    for tok in tokens:
        if 0 <= tok <= 127:
            active.add(tok)
        elif 128 <= tok <= 255:
            active.discard(tok - 128)
    if not active:
        return list(tokens)
    result = list(tokens)
    result.append(268)
    for pitch in sorted(active):
        result.append(128 + pitch)
    return result

# ── Generate with retry on degenerate output ──────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)

for attempt in range(1, MAX_RETRIES + 1):
    attempt_temp = TEMPERATURE * (1.0 + 0.3 * (attempt - 1))
    tokens     = _generate_piece(
        model, composer_map[COMPOSER], device,
        MAX_TOKENS, MIN_TOKENS, attempt_temp, TOP_K, TOP_P,
        context_window=512,
        composer_name=COMPOSER if (use_theory or use_era) else None,
        use_era=use_era,
        use_theory=use_theory,
        primer_tokens=primer_tokens,
    )
    note_count = sum(1 for t in tokens if 0 <= t <= 127)
    print(f'Attempt {attempt} (T={attempt_temp:.2f}): {len(tokens)} tokens, {note_count} NOTE_ON events')
    if note_count >= 10:
        break
    if attempt < MAX_RETRIES:
        print('  Too few notes — retrying with higher temperature...')

if note_count < 10:
    print('Warning: very few notes after retries — try increasing TEMPERATURE.')

events_to_midi(_finalize_tokens(tokens), OUTPUT)
print(f'\nGenerated {len(tokens)} tokens → {OUTPUT}')
print('Find it in your Drive or download from the Files panel on the left.')
